# wiktionary-de-parser — demo

Extract structured data (IPA, hyphenation, inflection, POS, lemma
references, rhymes, meanings) from a German Wiktionary XML dump.

Run `uv sync` first, then `uv run jupyter notebook`.

## 1. Locate the dump

Point at an existing local `.xml.bz2`, or pass `dump_dir_path=...` and
call `download_dump()` to fetch the latest dump on first run.

In [ ]:
from wiktionary_de_parser import WiktionaryDump, WiktionaryParser

dump = WiktionaryDump(
    dump_file_path="../tmp/dewiktionary-latest-pages-articles-multistream.xml.bz2"
)
# Or download it:
# dump = WiktionaryDump(dump_dir_path="../tmp")
# dump.download_dump()

parser = WiktionaryParser()

## 2. Parse a single page

One page can hold several entries (one per language and part of
speech), so `entries()` yields a `WiktionaryEntry` per Wortart section
and `parse()` turns each into a `ParsedEntry`.

In [ ]:
from pprint import pprint

for page in dump.pages():
    if page.redirect_to or not page.wikitext:
        continue
    if page.name == "Abend":
        for entry in parser.entries(page):
            pprint(parser.parse(entry))
        break

## 3. Read individual fields

`ParsedEntry` is a plain dataclass — access fields directly.

In [ ]:
for page in dump.pages():
    if page.redirect_to or not page.wikitext:
        continue
    if page.name == "Abend":
        entry = next(parser.entries(page))
        result = parser.parse(entry)
        print("lemma:        ", result.lemma)
        print("language:     ", result.language, result.language_code)
        print("pos:          ", result.pos)
        print("ipa:          ", result.ipa)
        print("hyphenation:  ", result.hyphenation)
        print("rhymes:       ", result.rhymes)
        print("inflection:   ", result.inflection)
        print("meanings:     ", result.meanings)
        break

## 4. Process the whole dump in parallel

`iter_parsed(workers=N)` keeps XML iteration on the main process and
shards parsing across a process pool. `workers` defaults to
`os.cpu_count() - 1`; pass `workers=1` to disable multiprocessing.

In [ ]:
count = 0
with_ipa = 0

for result in dump.iter_parsed(workers=4):
    count += 1
    if result.ipa:
        with_ipa += 1
    if count >= 50_000:  # stop early for the demo
        break

print(f"parsed {count} entries, {with_ipa} with IPA")